# Memory experiment - rotated surface code with a data-qubit defect

This notebook builds the defect protocol, visualizes the detector evolution, and runs circuit-level Monte Carlo simulation through LightStim's `SimulationPipeline`. The center data qubit is measured and disabled; neighboring checks become alternating X- and Z-type gauge measurements while unaffected checks remain active. Batch sweeps use [`benchmarks/memory/run_memory.py`](../../benchmarks/memory/run_memory.py).

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "lightstim").is_dir() and (path / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from lightstim.noise.config import NoiseConfig
from lightstim.plot.styles import PALETTE, apply_paper_style, bold_ticks
from lightstim.protocols.rotated_surface_defect import (
    RotatedSurfaceDefectMemoryExperiment,
)
from lightstim.simulation.decoder_backend import DecoderConfig, SimulationPipeline

In [ ]:
def uniform_noise(p):
    return NoiseConfig(
        p_idle=p, p_1q=p, p_2q=p, p_meas=p, p_reset=p
    )


def build_defect_memory(
    distance,
    *,
    p=None,
    pre_defect_rounds=None,
    post_defect_schedule=None,
):
    experiment = RotatedSurfaceDefectMemoryExperiment(
        distance=distance,
        pre_defect_rounds=pre_defect_rounds,
        post_defect_schedule=post_defect_schedule,
        noise_params=None if p is None else uniform_noise(p),
        noise_model="circuit_level",
    )
    return experiment, experiment.build()

## 1. Build a compact defect protocol

The default experiment uses `d` pre-defect rounds and `d` alternating post-defect rounds. The compact circuit below keeps one pre-defect round and two post-defect rounds so the complete transition fits in one detector-slice view.

In [ ]:
experiment, clean_circuit = build_defect_memory(
    3,
    pre_defect_rounds=1,
    post_defect_schedule=("Z", "X"),
)

affected_checks = [
    {
        "uid": uid,
        "basis": experiment.system.stabilizers[uid]["type"],
        "syndrome_coord": experiment.system.stabilizers[uid]["syn_coord"],
        "effective_weight": len(
            experiment.system.effective_stabilizer(uid)["data_indices"]
        ),
    }
    for uid in sorted(experiment.defect.affected_stabilizers)
]

noiseless_sample = clean_circuit.compile_detector_sampler().sample(
    32, append_observables=True
)
assert not noiseless_sample.any()

print("defect coordinate:", experiment.defect_coord)
print("gauge schedule:", experiment.gauge_schedule)
print("affected checks:", affected_checks)
print(
    f"qubits={clean_circuit.num_qubits}, "
    f"detectors={clean_circuit.num_detectors}, "
    f"observables={clean_circuit.num_observables}"
)

## 2. Visualize the detector evolution

The full compact circuit shows the ordinary checks before the defect, the center-qubit readout, both alternating gauge rounds, and the final memory readout.

In [ ]:
clean_circuit.without_noise().diagram("detslice-with-ops-svg")

## 3. Run the LightStim simulation pipeline

This fixed-shot sweep uses circuit-level noise and the public `SimulationPipeline` with the CPU PyMatching backend. It exercises the same sampling and decoding path used by the benchmark runner; the notebook does not manually construct or decode a detector error model.

In [ ]:
P_VALUES = (2e-3, 3e-3, 5e-3)
DISTANCES = (3, 5, 7)
SHOTS = 20_000

pipeline = SimulationPipeline(
    decoder_config=DecoderConfig("pymatching", backend="cpu"),
    max_shots=SHOTS,
    max_errors=SHOTS + 1,
    batch_size=SHOTS,
    num_workers=1,
    print_progress=False,
)

rows = []
for p in P_VALUES:
    for distance in DISTANCES:
        _, noisy_circuit = build_defect_memory(distance, p=p)
        metadata = {
            "code": "rotated_sc_defect",
            "distance": distance,
            "p": p,
            "decoder_name": "pymatching",
        }
        stats = pipeline.run(noisy_circuit, metadata)
        rows.append({
            **metadata,
            "shots": stats.shots,
            "errors": stats.errors,
            "logical_error_rate": stats.logical_error_rate,
            "plot_ler": max(stats.errors, 0.5) / stats.shots,
            "seconds": stats.seconds,
        })

results = pd.DataFrame(rows)
display(results.drop(columns="plot_ler"))

## 4. Visualize logical-error suppression

This compact sweep is a functional validation rather than a threshold study. Below threshold, increasing distance should suppress the logical error rate even though the center data qubit has been disabled.

In [ ]:
apply_paper_style()
fig, ax = plt.subplots(figsize=(5.4, 4.0))

for color, (p, group) in zip(PALETTE, results.groupby("p")):
    ax.semilogy(
        group["distance"],
        group["plot_ler"],
        marker="o",
        color=color,
        label=f"p={p:g}",
    )

ax.set_xticks(DISTANCES)
ax.set_xlabel("Code distance")
ax.set_ylabel("Logical error rate")
ax.set_title("Rotated surface-code memory with one data defect")
ax.legend()
bold_ticks(ax)
fig.tight_layout(pad=0.4)
plt.show()